In [31]:
import os
import warnings
import numpy as np
import tensorflow as tf
import gc

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")

np.random.seed(42)
tf.random.set_seed(42)


gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except:
        pass


train_dir = r"C:/Users/rkkol/4yearproject/datasets/lung/train"
val_dir = r"C:/Users/rkkol/4yearproject/datasets/lung/validation"
model_save_path = r"C:/Users/rkkol/4yearproject/models/ctscan_mobilenet.keras"

IMG_SIZE = (150, 150)
BATCH_SIZE = 8
NUM_CLASSES = 3


def create_data_generators():
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True
    )

    val_datagen = ImageDataGenerator(rescale=1./255)

    train_gen = train_datagen.flow_from_directory(
        train_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=True
    )

    val_gen = val_datagen.flow_from_directory(
        val_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    return train_gen, val_gen


def create_model():
    base_model = MobileNetV2(
        input_shape=(150, 150, 3),
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False  
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs)
    return model, base_model

def get_callbacks():
    return [
        ModelCheckpoint(model_save_path, monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, verbose=1)
    ]


def main():
    gc.collect()
    tf.keras.backend.clear_session()

    train_gen, val_gen = create_data_generators()

    print("Class mapping:", train_gen.class_indices)

    model, base_model = create_model()

    model.compile(
        optimizer=Adam(learning_rate=3e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    print("\n🔹 Training (feature extraction)...")
    model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=15,
        callbacks=get_callbacks(),
        verbose=1
    )

  
    print("\n🔹 Fine-tuning top layers...")
    base_model.trainable = True

    for layer in base_model.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=10,
        callbacks=get_callbacks(),
        verbose=1
    )


    loss, acc = model.evaluate(val_gen, verbose=1)
    print(f"\n Final Validation Accuracy: {acc:.4f}")

    model.save(model_save_path)
    print(f"Model saved at: {model_save_path}")

if __name__ == "__main__":
    main()


Found 889 images belonging to 3 classes.
Found 209 images belonging to 3 classes.
Class mapping: {'Bengin': 0, 'Malignant': 1, 'Normal': 2}
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 150, 150, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 75, 75, 32)        │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 75, 75, 32)        │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 75, 75, 32)        │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 75, 75, 32)        │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 75, 75, 32)        │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 75, 75, 32)        │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 75, 75, 16)        │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 75, 75, 16)        │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 75, 75, 96)        │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 75, 75, 96)        │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 75, 75, 96)        │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 77, 77, 96)        │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 38, 38, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,422,851 (9.24 MB)

 Trainable params: 164,611 (643.01 KB)

 Non-trainable params: 2,258,240 (8.61 MB)


🔹 Training (feature extraction)...
Epoch 1/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.5834 - loss: 1.1029
Epoch 1: val_accuracy improved from None to 0.81340, saving model to C:/Users/rkkol/4yearproject/models/ctscan_mobilenet.keras

Epoch 1: finished saving model to C:/Users/rkkol/4yearproject/models/ctscan_mobilenet.keras
112/112 ━━━━━━━━━━━━━━━━━━━━ 16s 108ms/step - accuracy: 0.6614 - loss: 0.8647 - val_accuracy: 0.8134 - val_loss: 0.7823 - learning_rate: 3.0000e-04
Epoch 2/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.8128 - loss: 0.5251
Epoch 2: val_accuracy improved from 0.81340 to 0.82297, saving model to C:/Users/rkkol/4yearproject/models/ctscan_mobilenet.keras

Epoch 2: finished saving model to C:/Users/rkkol/4yearproject/models/ctscan_mobilenet.keras
112/112 ━━━━━━━━━━━━━━━━━━━━ 11s 98ms/step - accuracy: 0.8076 - loss: 0.5342 - val_accuracy: 0.8230 - val_loss: 0.9686 - learning_rate: 3.0000e-04
Epoch 3/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step